# 🧭 Credit Card Fraud · S6E11 · Beginner → Expert, every step measured

> **One notebook, three levels.** Start at the top and stop whenever you like: each part ends with a
> valid submission and a CV score you can trust.

We start from a logistic regression and climb, **one idea at a time**, to a blend in the competitive range.
Every step is scored on **the same 5 folds**, so you can see exactly what each idea is worth, and what is
only noise. Everything is trained here, from scratch, on CPU. No external prediction files.

> 🆕 **Version 4 (Nov 4): optional [Part D](#partD) pools 72 public models with ours, the way the top of the board
> is built: nested CV 0.95214, public LB 0.95200.** Every source is credited, and the code checks that each one used
> our folds. Everything before Part D is still trained here, from scratch.

> **Version 3 (Nov 4).** New step 6: **boosting from a GLM**. Small trees start from a logistic regression's
> logit and fix only what it gets wrong. The best single model goes from 0.95051 to 0.95106, and the final blend
> from 0.95084 to 0.95110. The GLM got stronger too, by being solved to convergence ([section 9](#s9)). XGBoost
> left the blend because it added nothing, which keeps the run time about the same. [Section 11](#s11) now
> measures the public-board noise of every step-to-step gain, as asked in the comments. Thank you for the
> upvotes and questions!

> **TL;DR: four ideas that drive the score**
> 1. The labels follow a near-**logistic formula**, so tiny trees (3–4 leaves) and linear models beat deep models. ([section 5](#s5))
> 2. The generator **reuses values**, and each value has its own fraud rate. Encode it, out of fold. ([section 6](#s6))
> 3. The generator **writes numbers digit by digit**, and the digits carry the label. Encode them too. ([section 7](#s7))
> 4. A **GLM** gets the additive part right; **small trees that start from its logit** fix the rest. ([section 9](#s9))

### 📈 The score ladder (5-fold out-of-fold ROC AUC, produced by this notebook)

| Step | Level | What changes | CV AUC | Gain |
|:-:|:-:|---|:-:|:-:|
| 0 | 🟢 | Logistic regression on the raw columns | 0.94187 | |
| 1 | 🟡 | LightGBM with default-size trees (31 leaves) | 0.94663 | +0.00476 |
| 2 | 🟡 | The same LightGBM with **4 leaves** | 0.94771 | +0.00108 |
| 3 | 🟡 | + **per-value target encoding** | 0.94884 | +0.00113 |
| 4 | 🔴 | + **digit keys**: how each number is *written* | 0.95031 | +0.00147 |
| 5 | 🔴 | + **seed bagging** of the encodings | 0.95051 | +0.00019 |
| 6 | 🔴 | 🆕 **Boosting from a GLM**: 4-leaf trees that start at a logistic regression's logit | 0.95106 | +0.00055 |
| 7 | 🔴 | **Stacked rank blend** of steps 6, 5 and 2 | **0.95110** | +0.00004 |
| 8 | 🏆 | Optional: **pooled stack** of 72 public models + ours (Part D) | **0.95214** | +0.00104 |

**Public leaderboard: 0.95110** for the self-trained step-7 blend, the same as its CV (version 2's blend: 0.95083). Section 11
shows how far a public score can drift from CV.

### 🗺️ Choose your path

| If you are… | Read | You will learn |
|---|---|---|
| 🟢 **New to Kaggle** | [Part A](#partA), sections 1–4 | what ROC AUC rewards, five EDA facts that matter, honest cross-validation, your first submission |
| 🟡 **Comfortable with GBDTs** | [Part B](#partB), sections 5–6 | why *smaller* trees win here, and target encoding without fooling yourself |
| 🔴 **Chasing the top** | [Part C](#partC), sections 7–11 | the generator's digit fingerprint, seed bagging, honest blending, how noisy the public board is |
| 🏆 **Want the top-of-board recipe** | [Part D](#partD) | pooling public predictions honestly: same folds, nested CV, what each source adds |

<a id="contents"></a>
### 📑 Contents

0. [Setup](#setup)

**Part A 🟢**: 1. [The task in one minute](#s1) · 2. [Load and check the data](#s2) · 3. [EDA: five facts that matter](#s3) · 4. [First model and honest CV](#s4)

**Part B 🟡**: 5. [Gradient boosting, and why smaller trees win](#s5) · 6. [Per-value target encoding, and the leak that fools everyone](#s6)

**Part C 🔴**: 7. [The generator's fingerprint: digits carry the label](#s7) · 8. [Seed bagging](#s8) · 9. [Boosting from a GLM 🆕](#s9) · 10. [Blending, judged honestly](#s10) · 11. [How noisy is the public leaderboard?](#s11)

**Part D 🏆** (optional): [Pooling public predictions](#partD)

**Wrap-up**: 12. [From AUC to a fraud team's review queue](#s12) · 13. [What didn't work, and what to try next](#s13) · 14. [Credits](#s14) · 15. [The ladder, recomputed live](#s15)

⏱️ A full run takes about an hour on Kaggle's CPU, most of it in the seed bags of Part C.
For a quick first run, set `BAG_SEEDS = [0]` and `RUN_TREE_SWEEP = False` in the setup cell.

<a id="setup"></a>
## 0 · Setup

Every knob lives in `CFG`. The defaults reproduce the ladder above. For a quick first run, set
`BAG_SEEDS = [0]` and `RUN_TREE_SWEEP = False`.

In [ ]:
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgb
import sklearn
from IPython.display import display
from scipy.special import logit
from scipy.stats import rankdata
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import KFold, StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 40)


class CFG:
    SEED = 42                 # the outer 5-fold split, shared by every model in the notebook
    N_FOLDS = 5
    BAG_SEEDS = [0, 1, 2]     # Part C seed bags; [0] runs about 3x faster
    TE_SMOOTHING = 20         # pseudo-count m of the target encoding (section 6)
    TE_INNER_FOLDS = 5        # inner cross-fitting folds of the target encoding
    RUN_TREE_SWEEP = True     # tree-size sweep on fold 0 (section 5), a few minutes
    THREADS = os.cpu_count()


def find_file(name):
    """Find an input file on Kaggle (/kaggle/input/...) or in a local ../data folder."""
    for root in (Path("/kaggle/input"), Path("../data"), Path("data")):
        if root.exists():
            hits = sorted(root.rglob(name))
            if hits:
                return hits[0]
    return None


COMP_DIR = find_file("sample_submission.csv").parent
ORIGINAL_CSV = find_file("credit_card_fraud_2026.csv")   # optional: the dataset this one was generated from
print("competition data:", COMP_DIR)
print("original data   :", ORIGINAL_CSV or "not attached (section 5.3 will be skipped)")
print(f"lightgbm {lgb.__version__} · scikit-learn {sklearn.__version__} · pandas {pd.__version__} · {CFG.THREADS} CPU threads")
T_START = time.time()

In [ ]:
# A quiet, consistent chart style: one accent colour, hairline grid, ink-coloured text.
SURFACE, INK, INK_2, MUTED, GRID, AXIS = "#fcfcfb", "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
BLUE, RED, GRAY = "#2a78d6", "#e34948", "#c3c2b7"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": AXIS, "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 11, "axes.titleweight": "bold", "axes.axisbelow": True, "axes.titlecolor": INK, "axes.titlelocation": "left",
    "axes.labelcolor": INK_2, "axes.labelsize": 9.5, "axes.grid": True,
    "grid.color": GRID, "grid.linewidth": 0.8, "grid.linestyle": "-",
    "xtick.color": AXIS, "ytick.color": AXIS, "xtick.labelcolor": INK_2, "ytick.labelcolor": INK_2,
    "xtick.labelsize": 9, "ytick.labelsize": 9, "legend.frameon": False, "legend.fontsize": 9,
    "lines.linewidth": 2, "lines.solid_capstyle": "round", "font.size": 10, "figure.dpi": 110,
})


def suptitle(fig, text):
    fig.suptitle(text, x=0.01, ha="left", fontsize=13, fontweight="bold", color=INK)

<a id="partA"></a>
# 🟢 Part A · Beginner

<a id="s1"></a>
## 1 · The task in one minute

Each row is one card transaction (amount, merchant, device, CVV retries, ...), 24 columns in all.
We predict `is_fraud`: the **probability** that the transaction is fraudulent.

**The metric is ROC AUC.** Pick one random fraud and one random legitimate transaction. AUC is the
probability that your model gives the fraud the higher score.

- 0.5 is a coin flip and 1.0 is perfect.
- Only the **order** of your predictions matters, not their calibration. You could even submit ranks,
  and Part C does exactly that when blending.

**Why not accuracy?** Fraud is rare (about 1.7% of rows). A "model" that always answers *not fraud* is
98.3% accurate and catches nothing. AUC can't be fooled that way: that model scores exactly 0.5.

<a id="s2"></a>
## 2 · Load and check the data

In [ ]:
train = pd.read_csv(COMP_DIR / "train.csv")
test = pd.read_csv(COMP_DIR / "test.csv")
TARGET, ID = "is_fraud", "id"
FEATURES = [c for c in test.columns if c != ID]
BOOL = [c for c in FEATURES if train[c].dtype.kind == "b"]
NUM = [c for c in FEATURES if train[c].dtype.kind in "if"]
CAT = [c for c in FEATURES if c not in BOOL + NUM]
y = train[TARGET].astype(int).to_numpy()

print(f"train {train.shape} · test {test.shape}")
print(f"{len(NUM)} numeric · {len(BOOL)} yes/no flags · {len(CAT)} categorical")
print(f"missing values: train {train.isna().sum().sum()}, test {test.isna().sum().sum()}")
print(f"duplicate feature rows: train {train[FEATURES].duplicated().sum()}, test {test[FEATURES].duplicated().sum()}")
print(f"fraud: {y.sum():,} of {len(y):,} rows = {y.mean():.2%}")
print(f"accuracy of always answering 'not fraud': {1 - y.mean():.2%}")
display(train.head())

No missing values, no duplicates, nothing to clean. 11,689 frauds in 699,960 rows: rare, but plenty
to learn from.

<a id="s3"></a>
## 3 · EDA: five facts that matter

A good EDA answers one question: *what should the model pay attention to?* Here are the five facts that
shaped every model below.

### Fact 1: two categoricals matter, three don't

In [ ]:
base = y.mean()
fig, axes = plt.subplots(1, len(CAT), figsize=(18, 4.2), sharex=True)
for ax, col in zip(axes, CAT):
    rate = train.groupby(col)[TARGET].mean().sort_values() * 100
    ax.barh(rate.index.astype(str), rate.values, height=0.6, color=BLUE)
    ax.axvline(base * 100, color=INK_2, lw=1)
    ax.set_title(col)
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("fraud rate, %")
suptitle(fig, f"Fraud rate by category (vertical line: all rows, {base:.2%}). "
              "Merchant and authentication matter; card, channel and device are flat")
plt.tight_layout()
plt.show()

Gift cards, crypto exchanges and gaming run at 3–4x the base rate. *No authentication* is 6.4% fraud,
biometric only 0.5%. Card type, channel and device all sit between 1.5% and 1.9%: on their own they say
almost nothing.

### Fact 2: red flags compound

Six yes/no flags (foreign transaction, VPN, IP-country mismatch, ...). Count how many are set on a row:

In [ ]:
n_flags = train[BOOL].sum(axis=1).clip(upper=4)
flags = train.groupby(n_flags)[TARGET].agg(["mean", "size"])
flags.index = [str(i) if i < 4 else "4+" for i in flags.index]

fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(flags.index, flags["mean"] * 100, width=0.55, color=BLUE)
for i, (rate, n) in enumerate(zip(flags["mean"], flags["size"])):
    ax.text(i, rate * 100 + 2, f"{rate:.1%}\n{n:,} rows", ha="center", fontsize=8.5, color=INK_2)
ax.set_ylim(0, 105)
ax.set_xlabel("number of red flags set")
ax.set_ylabel("fraud rate, %")
ax.grid(axis="x", visible=False)
ax.set_title("Each extra flag multiplies the fraud rate")
plt.tight_layout()
plt.show()

0.7% with no flag, 3.6% with one, 18% with two, 52% with three: every extra flag multiplies the *odds* of
fraud by roughly 5. **Effects that multiply the odds are effects that add on the log-odds scale**, and that
is exactly what a logistic model assumes. Remember this; it explains Part B.

### Fact 3: the numeric signal sits in a few columns, as steady staircases

In [ ]:
uni = pd.Series({c: max(roc_auc_score(y, train[c]), 1 - roc_auc_score(y, train[c])) for c in NUM}).sort_values()

fig = plt.figure(figsize=(17, 4.6))
grid = fig.add_gridspec(1, 5, width_ratios=[2.2, 1, 1, 1, 1])
ax = fig.add_subplot(grid[0])
ax.barh(uni.index, uni.values - 0.5, left=0.5, height=0.6, color=[BLUE if v > 0.55 else GRAY for v in uni.values])
for i, v in enumerate(uni.values):
    ax.text(v + 0.003, i, f"{v:.3f}", va="center", fontsize=8.5, color=INK_2)
ax.set_xlim(0.5, 0.79)
ax.set_xlabel("AUC of the raw column alone (best direction)")
ax.grid(axis="y", visible=False)
ax.set_title("One column at a time")

top4 = uni.index[::-1][:4]
for k, col in enumerate(top4):
    a = fig.add_subplot(grid[k + 1])
    groups = train[col] if train[col].nunique() <= 12 else pd.qcut(train[col], 10, duplicates="drop")
    g = train.groupby(groups, observed=True)[TARGET].agg(["mean", "size"])
    g = g[g["size"] >= 500]
    a.plot(range(len(g)), g["mean"] * 100, color=BLUE, marker="o", markersize=4.5,
           markeredgecolor=SURFACE, markeredgewidth=1.2)
    a.set_title(col, fontsize=9.5)
    a.set_xlabel("value" if train[col].nunique() <= 12 else "decile", fontsize=8.5)
    if train[col].nunique() <= 12:
        a.set_xticks(range(len(g)), [str(int(v)) for v in g.index])
    else:
        a.set_xticks([0, len(g) - 1], ["low", "high"])
    a.grid(axis="x", visible=False)
    if k == 0:
        a.set_ylabel("fraud rate, %")
suptitle(fig, "Merchant risk, velocity and CVV retries carry most of the numeric signal")
plt.tight_layout()
plt.show()

print("Fraud rate by CVV retries:")
display(train.groupby("cvv_retry_count")[TARGET].agg(fraud_rate="mean", rows="size")
        .style.format({"fraud_rate": "{:.2%}", "rows": "{:,}"}))

`merchant_risk_score`, `velocity_score` and `cvv_retry_count` stand out (AUC 0.71–0.75 alone), then
`customer_age` and `txn_count_last_24h`. One CVV retry is 8.8% fraud; two retries are 42%.
Hour, weekday, distance, card age and balance look flat. Keep them anyway: section 6 shows that
"flat" columns can still hide signal.

### Fact 4: train and test come from the same distribution

**Adversarial validation**: label train rows 0 and test rows 1, and ask a model to tell them apart.
If it can't (AUC ≈ 0.5), the test set looks like the training set, and our CV score should carry over.

In [ ]:
both = pd.concat([train[FEATURES], test[FEATURES]], ignore_index=True)
for c in CAT:
    both[c] = both[c].astype("category")
is_test = np.r_[np.zeros(len(train)), np.ones(len(test))]
adv = cross_val_predict(lgb.LGBMClassifier(n_estimators=200, learning_rate=0.1, verbose=-1, n_jobs=CFG.THREADS),
                        both, is_test, cv=3, method="predict_proba")[:, 1]
print(f"adversarial validation AUC (train vs test): {roc_auc_score(is_test, adv):.4f}")
del both, adv

0.500: a model can't tell train from test. **Plain stratified K-fold CV is a reliable judge here.**

### Fact 5: rules alone would miss a third of the fraud

The flags are strong, so why not just write rules? Because a large share of fraud sets **no flag at all**:

In [ ]:
no_flag = train[BOOL].sum(axis=1) == 0
print(f"rows with no flag: {no_flag.mean():.1%} of all rows, fraud rate {train.loc[no_flag, TARGET].mean():.2%}")
print(f"frauds with no flag: {y[no_flag.to_numpy()].sum():,} of {y.sum():,} = {y[no_flag.to_numpy()].sum() / y.sum():.0%} of all fraud")

A third of all frauds hide among the 78% of transactions with no flag. Finding them means combining many weak
signals at once, which is the job of a model.

> 💡 **What we carry into modeling**# > - The signal is concentrated: merchant risk, velocity, CVV retries, authentication, merchant category, the six flags.
> - Effects look like staircases that *add up on the log-odds scale*, the signature of a logistic formula.
> - Train and test match, so a stratified 5-fold CV is a trustworthy judge.

<a id="s4"></a>
## 4 · First model: logistic regression, and honest cross-validation

**Cross-validation is the habit that matters most.** We split the training rows into 5 folds. Each fold takes
a turn as the validation set while a model trains on the other four. Every training row then gets exactly
one prediction from a model that never saw it: its **out-of-fold (OOF)** prediction. The AUC over all OOF
predictions is our **CV score**. The average of the 5 fold models' predictions on the test set is our
submission.

The split is **stratified**: with only 1.67% fraud, a purely random split could give folds with noticeably
different fraud rates, so we keep the rate identical in every fold. And every model in this notebook uses
**the same 5 folds**. That is what makes the ladder a fair comparison.

In [ ]:
FOLDS = list(StratifiedKFold(CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED).split(train, y))
LADDER = []          # one row per step of the ladder
OOF, PRED = {}, {}   # out-of-fold and test predictions of every model, by name


def log_step(step, name, what, oof, pred):
    """Store a model's predictions and add its CV score to the ladder."""
    OOF[name], PRED[name] = oof, pred
    score = roc_auc_score(y, oof)
    prev = LADDER[-1]["CV AUC"] if LADDER else None
    LADDER.append({"Step": step, "Model": name, "What changes": what, "CV AUC": score})
    gain = "" if prev is None else f"   ({score - prev:+.5f} vs step {LADDER[-2]['Step']})"
    print(f"Step {step} · {what}: CV AUC {score:.5f}{gain}")


def save_submission(pred, path):
    pd.DataFrame({ID: test[ID], TARGET: pred}).to_csv(path, index=False)
    print(f"saved {path} ({len(pred):,} rows)")


def cv_sklearn(make_model, X, X_test):
    """5-fold CV for any scikit-learn model: returns (OOF predictions, mean test prediction)."""
    oof, pred = np.zeros(len(X)), np.zeros(len(X_test))
    for k, (tr, va) in enumerate(FOLDS):
        model = make_model().fit(X.iloc[tr], y[tr])
        oof[va] = model.predict_proba(X.iloc[va])[:, 1]
        pred += model.predict_proba(X_test)[:, 1] / len(FOLDS)
        print(f"  fold {k}: AUC {roc_auc_score(y[va], oof[va]):.5f}")
    return oof, pred

The model: numbers scaled to mean 0 and standard deviation 1, categories one-hot encoded, and a logistic
regression on top. The preprocessing sits **inside** a pipeline, so it is fitted on each training fold only.
Fitting a scaler on all rows before splitting is a small leak; with target-based features (section 6) the
same mistake becomes a huge one.

In [ ]:
X_LR, T_LR = train[FEATURES].copy(), test[FEATURES].copy()
X_LR[BOOL], T_LR[BOOL] = X_LR[BOOL].astype(int), T_LR[BOOL].astype(int)


def make_logreg():
    prep = ColumnTransformer([("num", StandardScaler(), NUM + BOOL),
                              ("cat", OneHotEncoder(handle_unknown="ignore"), CAT)])
    return make_pipeline(prep, LogisticRegression(max_iter=2000))


oof, pred = cv_sklearn(make_logreg, X_LR, T_LR)
log_step(0, "logreg_raw", "Logistic regression, raw columns", oof, pred)
save_submission(pred, "submission_A_logreg.csv")

**0.942 with a linear model.** That is already strong, and a hint about how this data was built.

> 📤 **Your first submission.** Click *Save Version* and run the notebook. When it finishes, open the
> *Output* tab, pick a `submission_*.csv` file and click *Submit*. You get 5 submissions a day. At the end of
> the competition you choose 2 for the final (private) leaderboard, so keep notes of each one's CV score.

⏭️ That is the whole workflow. Everything from here on improves the model inside it.
[Back to contents](#contents)

---
<a id="partB"></a>
# 🟡 Part B · Intermediate

<a id="s5"></a>
## 5 · Gradient boosting, and why *smaller* trees win

### 5.1 LightGBM with default-size trees

Gradient-boosted trees are the default tool for tabular data. LightGBM takes the categoricals as they are
(pandas `category` dtype) and needs no scaling. Early stopping on the validation fold chooses the number of
trees.

> 🔍 *Expert note:* early stopping peeks at the validation fold, which flatters the OOF score by a hair.
> Every step here does it the same way, so the comparisons stay fair.

In [ ]:
CATEGORIES = {c: sorted(pd.concat([train[c], test[c]]).unique()) for c in CAT}


def model_frame(df):
    """Raw columns, ready for LightGBM / XGBoost: categoricals as pandas categories, flags as 0/1."""
    out = df[FEATURES].copy()
    for c in CAT:
        out[c] = pd.Categorical(out[c], categories=CATEGORIES[c])
    out[BOOL] = out[BOOL].astype(int)
    return out


X_RAW, T_RAW = model_frame(train), model_frame(test)

LGB_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=4, min_data_in_leaf=100,
                  feature_fraction=0.5, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                  verbose=-1, num_threads=CFG.THREADS)


def fit_lgb(params, X_tr, y_tr, X_va, y_va, seed=0):
    d_tr = lgb.Dataset(X_tr, y_tr)
    d_va = lgb.Dataset(X_va, y_va, reference=d_tr)
    return lgb.train({**params, "seed": seed}, d_tr, num_boost_round=30_000, valid_sets=[d_va],
                     callbacks=[lgb.early_stopping(300, verbose=False)])


def cv_lgb(build_fold, params=LGB_PARAMS, seeds=(0,)):
    """5-fold CV for LightGBM, averaged over seeds.

    build_fold(tr, va, seed) returns (X_train, X_valid, X_test) for one fold. Anything that learns from
    labels (like the target encoding of section 6) must happen inside it, on the training rows only.
    """
    oof, pred = np.zeros(len(train)), np.zeros(len(test))
    for seed in seeds:
        for k, (tr, va) in enumerate(FOLDS):
            X_tr, X_va, X_te = build_fold(tr, va, seed)
            booster = fit_lgb(params, X_tr, y[tr], X_va, y[va], seed)
            p_va = booster.predict(X_va, num_iteration=booster.best_iteration)
            oof[va] += p_va / len(seeds)
            pred += booster.predict(X_te, num_iteration=booster.best_iteration) / (len(FOLDS) * len(seeds))
            print(f"  seed {seed} fold {k}: AUC {roc_auc_score(y[va], p_va):.5f} · {booster.best_iteration} trees")
    return oof, pred


def raw_view(tr, va, seed):
    return X_RAW.iloc[tr], X_RAW.iloc[va], T_RAW

In [ ]:
oof, pred = cv_lgb(raw_view, {**LGB_PARAMS, "num_leaves": 31})
log_step(1, "lgb31_raw", "LightGBM, 31 leaves, raw columns", oof, pred)

A big jump over the linear model. The usual next move is bigger trees and more tuning. Here, going the
**other way** works better.

### 5.2 Tree size, measured on one fold

In [ ]:
if CFG.RUN_TREE_SWEEP:
    tr, va = FOLDS[0]
    rows = []
    for leaves in [2, 3, 4, 7, 15, 31, 63]:
        b = fit_lgb({**LGB_PARAMS, "num_leaves": leaves}, X_RAW.iloc[tr], y[tr], X_RAW.iloc[va], y[va])
        rows.append({"num_leaves": leaves, "fold-0 AUC": roc_auc_score(y[va], b.predict(X_RAW.iloc[va], num_iteration=b.best_iteration)),
                     "trees": b.best_iteration})
    sweep = pd.DataFrame(rows)
    best = sweep["fold-0 AUC"].idxmax()

    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(sweep["num_leaves"], sweep["fold-0 AUC"], color=BLUE, marker="o", markersize=6,
            markeredgecolor=SURFACE, markeredgewidth=1.5)
    ax.annotate(f"best: {sweep.loc[best, 'num_leaves']} leaves", (sweep.loc[best, "num_leaves"], sweep.loc[best, "fold-0 AUC"]),
                xytext=(12, -16), textcoords="offset points", fontsize=9, color=INK_2)
    ax.set_xscale("log", base=2)
    ax.set_xticks(sweep["num_leaves"], sweep["num_leaves"].astype(str))
    ax.minorticks_off()
    ax.set_xlabel("num_leaves (log scale)")
    ax.set_ylabel("AUC, fold 0")
    ax.set_title("Smaller trees score higher, down to 3-4 leaves")
    plt.tight_layout()
    plt.show()
    display(sweep.style.format({"fold-0 AUC": "{:.5f}"}).hide(axis="index"))

Unusual for tabular data: **3–4 leaves beat 31**, and even 2 leaves (one split per tree, no interactions
at all) beat 15 and above. Why?

### 5.3 Why: look at where the data came from

Playground data is generated by a model trained on a small real-looking dataset. Here that is
[Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026)
(20,000 rows). Two quick tests on it, if you have attached it to the notebook:

In [ ]:
if ORIGINAL_CSV is not None:
    orig = pd.read_csv(ORIGINAL_CSV)
    X_orig = orig[FEATURES].copy()
    X_orig[BOOL] = X_orig[BOOL].astype(int)
    cv5 = StratifiedKFold(5, shuffle=True, random_state=CFG.SEED)
    p_lr = cross_val_predict(make_logreg(), X_orig, orig[TARGET], cv=cv5, method="predict_proba")[:, 1]
    p_gb = cross_val_predict(lgb.LGBMClassifier(n_estimators=300, learning_rate=0.05, verbose=-1, n_jobs=CFG.THREADS),
                             model_frame(orig), orig[TARGET], cv=cv5, method="predict_proba")[:, 1]
    transfer = make_logreg().fit(X_orig, orig[TARGET])
    display(pd.DataFrame({
        "test": ["5-fold CV inside the original 20k rows", "5-fold CV inside the original 20k rows",
                 "fit on the original 20k rows, scored on our 700k training rows"],
        "model": ["logistic regression", "LightGBM, 31 leaves", "logistic regression"],
        "AUC": [roc_auc_score(orig[TARGET], p_lr), roc_auc_score(orig[TARGET], p_gb),
                roc_auc_score(y, transfer.predict_proba(X_LR)[:, 1])],
    }).style.format({"AUC": "{:.4f}"}).hide(axis="index"))
else:
    print("Attach 'uditjain13/credit-card-fraud-detection-2026' to run this cell.")

- Inside the original data, a plain **logistic regression beats LightGBM** by a wide margin (0.945 vs 0.921).
- A logistic regression fitted on just the 20k original rows scores **0.934 on our 700k rows**, without
  seeing a single one of them.

The labels behave like the output of a **logistic formula**: each feature adds its own push to the
log-odds, with few interactions. A model that matches that shape wins. A 4-leaf tree is close to additive.
A 31-leaf tree spends its splits on interactions that aren't there, and fits noise.

> 💡 **Key idea:** match the model's shape to the data's shape. Here that means *shallow*.

### 5.4 LightGBM with 4 leaves, all 5 folds

In [ ]:
oof, pred = cv_lgb(raw_view)
log_step(2, "lgb4_raw", "LightGBM, 4 leaves, raw columns", oof, pred)

<a id="s6"></a>
## 6 · Per-value target encoding, and the leak that fools everyone

### 6.1 Values repeat, a lot

The generator reuses values. `account_balance_usd` has 43k distinct values in 700k rows, and
`distance_from_home_km` only 4.4k. When a value shows up dozens of times, the fraud rate *of that exact
value* becomes something we can measure.

In [ ]:
reps = pd.DataFrame({"distinct values": train[NUM].nunique(),
                     "median rows per value": [train[c].value_counts().median() for c in NUM]})
display(reps.sort_values("distinct values", ascending=False).style.format("{:,.0f}"))

### 6.2 The encoding

**Target encoding** replaces each value by the fraud rate of the rows that share it. Rare values get
noisy rates (one row, one fraud: 100%!), so we *smooth* them toward the overall rate:

$$\text{encoding}(v) = \frac{\text{frauds with value } v \;+\; m \cdot \text{overall rate}}{\text{rows with value } v \;+\; m}, \qquad m = 20$$

A value seen 1,000 times keeps its own rate; a value seen twice stays close to the overall 1.67%.
We treat **every** column this way, numeric ones included: each distinct number is a category.

We need integer codes for each column (shared by train and test, no labels involved), then a few lines of
`numpy`. `np.bincount` counts rows and frauds per value in one pass, which keeps this fast enough to rerun
for every fold and seed.

In [ ]:
def factorize(frame_train, frame_test, cols):
    """Integer code of every value, shared by train and test. Uses no labels, so it is safe before the folds."""
    codes, n_levels = {}, {}
    for c in cols:
        k, uniques = pd.factorize(pd.concat([frame_train[c], frame_test[c]], ignore_index=True))
        k = k.astype(np.int32)                       # half the memory of pandas' int64 codes
        codes[c] = (k[:len(frame_train)], k[len(frame_train):])
        n_levels[c] = len(uniques)
    return codes, n_levels


CODES, N_LEVELS = factorize(train, test, FEATURES)


def te_table(codes, target, n_levels, m):
    """Smoothed fraud rate of every level, learned from (codes, target)."""
    frauds = np.bincount(codes, weights=target, minlength=n_levels)
    rows = np.bincount(codes, minlength=n_levels)
    return (frauds + m * target.mean()) / (rows + m)

### 6.3 ⚠️ The leak

The tempting version: compute every value's fraud rate on the **whole** training set, add it as a column,
then cross-validate. Let's try it on fold 0.

In [ ]:
tr, va = FOLDS[0]
leaky = pd.DataFrame({f"te_{c}": te_table(CODES[c][0], y.astype(float), N_LEVELS[c], CFG.TE_SMOOTHING)[CODES[c][0]]
                      for c in FEATURES}, index=X_RAW.index)
X_leaky = pd.concat([X_RAW, leaky], axis=1)
b = fit_lgb(LGB_PARAMS, X_leaky.iloc[tr], y[tr], X_leaky.iloc[va], y[va])
leak_auc = roc_auc_score(y[va], b.predict(X_leaky.iloc[va], num_iteration=b.best_iteration))
print(f"fold-0 AUC with the leaky encoding: {leak_auc:.5f}   <- too good to be true")
del leaky, X_leaky

**0.98: far above anything you'll see on the leaderboard.** Every validation row's encoding contains *its
own label*. For a value seen 3 times, one of those 3 is the row itself, so the encoding partly *is*
the answer. The test set has no labels, so none of this carries over, and the public score will be far below
the CV score. **Whenever CV looks too good to be true, look for this.**

### 6.4 The honest version: nested out-of-fold encoding

```
outer fold k:   [ ------------- training rows (80%) ------------- | validation rows (20%) ]

  training rows   -> inner 5-fold split of the training rows: each row is encoded by the other
                     4/5, so no row ever sees its own label
  validation rows -> encoded with all training rows of fold k (like the test set)
  test rows       -> encoded with all training rows of fold k
```

The model trains on encodings that are slightly noisy (computed without the row itself), exactly like the
ones it will meet on validation and test rows. `sklearn.preprocessing.TargetEncoder` implements the same
idea (`fit_transform` cross-fits internally). Ours is a few lines of `numpy`, with the smoothing in plain view.

In [ ]:
def target_encode_fold(cols, tr, va, seed, m=CFG.TE_SMOOTHING):
    """Out-of-fold target encoding of `cols` for one outer fold -> (train, valid, test) DataFrames."""
    y_tr = y[tr].astype(float)
    inner = list(KFold(CFG.TE_INNER_FOLDS, shuffle=True, random_state=seed).split(tr))
    E_tr = np.empty((len(tr), len(cols)), np.float32)
    E_va = np.empty((len(va), len(cols)), np.float32)
    E_te = np.empty((len(test), len(cols)), np.float32)
    for j, c in enumerate(cols):
        codes_train, codes_test = CODES[c]
        k_tr = codes_train[tr]
        for fit_idx, enc_idx in inner:                     # training rows: cross-fitted
            E_tr[enc_idx, j] = te_table(k_tr[fit_idx], y_tr[fit_idx], N_LEVELS[c], m)[k_tr[enc_idx]]
        full = te_table(k_tr, y_tr, N_LEVELS[c], m)        # validation + test rows: all training rows
        E_va[:, j], E_te[:, j] = full[codes_train[va]], full[codes_test]
    names = [f"te_{c}" for c in cols]
    return (pd.DataFrame(E_tr, columns=names, index=X_RAW.index[tr]),
            pd.DataFrame(E_va, columns=names, index=X_RAW.index[va]),
            pd.DataFrame(E_te, columns=names, index=T_RAW.index))


def te_view(cols, m=CFG.TE_SMOOTHING):
    """Raw columns + out-of-fold target encodings of `cols`."""
    def build_fold(tr, va, seed):
        E_tr, E_va, E_te = target_encode_fold(cols, tr, va, seed, m)
        return (pd.concat([X_RAW.iloc[tr], E_tr], axis=1), pd.concat([X_RAW.iloc[va], E_va], axis=1),
                pd.concat([T_RAW, E_te], axis=1))
    return build_fold

### 6.5 Does each value really carry its own risk?

For each numeric column: the AUC of the raw number, against the AUC of its per-value encoding, both out of
fold.

In [ ]:
def oof_encoding(col, m=CFG.TE_SMOOTHING):
    """Out-of-fold encoding of one column over the 5 outer folds (for diagnostics)."""
    out = np.zeros(len(y))
    codes_train = CODES[col][0]
    for tr, va in FOLDS:
        out[va] = te_table(codes_train[tr], y[tr].astype(float), N_LEVELS[col], m)[codes_train[va]]
    return out


cmp = pd.DataFrame({"raw value": uni,
                    "per-value encoding": {c: roc_auc_score(y, oof_encoding(c)) for c in NUM}})
cmp["gain"] = cmp["per-value encoding"] - cmp["raw value"]
cmp = cmp.sort_values("gain")

fig, ax = plt.subplots(figsize=(9, 4.6))
yy = np.arange(len(cmp))
ax.hlines(yy, cmp["raw value"], cmp["per-value encoding"], color=GRID, lw=2.5)
ax.scatter(cmp["raw value"], yy, s=40, color=GRAY, zorder=3, label="raw value", edgecolor=SURFACE, linewidth=1.5)
ax.scatter(cmp["per-value encoding"], yy, s=40, color=BLUE, zorder=3, label="per-value encoding", edgecolor=SURFACE, linewidth=1.5)
ax.set_yticks(yy, cmp.index)
ax.set_xlabel("AUC of the column alone (out of fold)")
ax.grid(axis="y", visible=False)
ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title("Per-value encoding finds signal in columns that look flat")
plt.tight_layout()
plt.show()
display(cmp.sort_values("gain", ascending=False).style.format("{:.4f}"))

`account_balance_usd` is useless as a number (AUC 0.506) but carries signal value by value (0.536).
`customer_age` and `hours_since_last_txn` gain too. No smooth function of the number can find this: the
generator gave individual values their own fraud rate. On the low-cardinality columns the encoding brings
nothing new, and the trees still have the raw values.

### 6.6 LightGBM with the encodings, all 5 folds

In [ ]:
oof, pred = cv_lgb(te_view(FEATURES))
log_step(3, "lgb4_te", "+ per-value target encoding (24 columns)", oof, pred)
save_submission(pred, "submission_B_lgb_te.csv")

> 💡 **Part B recap:** shallow trees (the data is close to a logistic formula), plus every value encoded by
> its own out-of-fold fraud rate. The leak demo is the most important lesson here: an encoding fitted
> outside the folds gave a fold-0 AUC of 0.98, a number no honest model gets near.

[Back to contents](#contents)

---
<a id="partC"></a>
# 🔴 Part C · Expert

<a id="s7"></a>
## 7 · The generator's fingerprint: digits carry the label

Section 6 treated each number as a *value*. Now look at how numbers are *written*. Here is the fraud rate by
the **first decimal digit** of four money/distance/time columns, as a gap from the overall rate:

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(17, 3.6), sharey=True)
two_dec = ["amount_usd", "hours_since_last_txn", "distance_from_home_km", "account_balance_usd"]
for ax, col in zip(axes, two_dec):
    digit = (np.rint(train[col].to_numpy() * 100).astype(np.int64) // 10) % 10
    gap = (train.groupby(digit)[TARGET].mean() - base) * 100
    ax.bar(gap.index, gap.values, width=0.6, color=[RED if v > 0 else BLUE for v in gap.values])
    ax.axhline(0, color=INK_2, lw=1)
    ax.set_xticks(range(10))
    ax.set_title(col, fontsize=10)
    ax.set_xlabel("first decimal digit")
    ax.grid(axis="x", visible=False)
axes[0].set_ylabel("fraud rate minus overall rate, pp")
suptitle(fig, "Fraud rate by first decimal digit: the same digits run hot (.0, .4) or cold (.3, .6) across columns")
plt.tight_layout()
plt.show()

A real distance does not care about its first decimal. Here the **same digits** run above the base rate
(`.0`, `.4`) or below it (`.3`, `.6`) in amounts, hours *and* distances (balances show a weaker version).
The likeliest explanation: the
generator writes each row like text, digit by digit, and its digit choices are tangled up with the label it
writes. Train and test were written by the same generator, so the pattern holds on the test rows too.

### 7.1 Digit keys

For each of the six columns with decimals, we cut the number into pieces and target-encode each piece
exactly like a value. For `1371.37`:

| key | value | groups together |
|---|---|---|
| `int` | 1371 | the integer part |
| `int10` | 137 | the integer part without its last digit |
| `tenths` | 13713 | the number cut to one decimal |
| `cents` | 37 | the two decimals |
| `lead1` | 4·1 | number of integer digits + first digit |
| `lead2` | 4·13 | number of integer digits + first two digits |

Two details matter. **Truncate, don't round**: `rint(x * 100)` only removes float error (0.29 × 100 =
28.999...), then integer division cuts. And the keys use no labels, so building them once on train + test is
safe; only their *encoding* happens inside the folds.

In [ ]:
FLOAT_COLS = [c for c in NUM if train[c].dtype.kind == "f"]


def digit_keys(df):
    """How each number is written: six label-free keys per float column."""
    keys = {}
    for c in FLOAT_COLS:
        units = np.rint(df[c].to_numpy() * 100).astype(np.int64)        # 1371.37 -> 137137
        integer = units // 100
        s = pd.Series(integer).astype(str)
        keys[f"{c}__int"] = integer                                       # 1371
        keys[f"{c}__int10"] = units // 1000                               # 137
        keys[f"{c}__tenths"] = units // 10                                # 13713
        keys[f"{c}__cents"] = units % 100                                 # 37
        keys[f"{c}__lead1"] = (s.str.len() * 10 + s.str[0].astype(int)).to_numpy()     # 4|1
        keys[f"{c}__lead2"] = (s.str.len() * 100 + s.str[:2].astype(int)).to_numpy()   # 4|13
    return pd.DataFrame(keys, index=df.index)


keys_train, keys_test = digit_keys(train), digit_keys(test)
KEYS = list(keys_train.columns)
key_codes, key_levels = factorize(keys_train, keys_test, KEYS)
CODES.update(key_codes)
N_LEVELS.update(key_levels)
ALL_KEYS = FEATURES + KEYS
print(f"{len(FLOAT_COLS)} float columns x 6 keys = {len(KEYS)} digit keys · {len(ALL_KEYS)} encoded columns in total")

# How much does one key say on its own? (out-of-fold AUC of its encoding)
key_auc = pd.DataFrame({c: {"raw number": uni[c], "exact value": cmp.loc[c, "per-value encoding"],
                            **{k: roc_auc_score(y, oof_encoding(f"{c}__{k}")) for k in ["int", "int10", "tenths", "cents", "lead1", "lead2"]}}
                        for c in two_dec}).T
display(key_auc.style.format("{:.3f}").background_gradient(cmap="Blues", axis=1))

Look at the `cents` column: the last two decimals of **distance from home** (0.52) and **hours since the last
transaction** (0.56) predict fraud on their own. For `amount_usd`, the integer part and the leading digits say
more than the exact value. The coarser keys pool many rows, so their rates are measured with less noise.

### 7.2 LightGBM with value and digit encodings

In [ ]:
oof, pred = cv_lgb(te_view(ALL_KEYS), seeds=CFG.BAG_SEEDS[:1])
log_step(4, "lgb4_digits", f"+ digit keys ({len(ALL_KEYS)} encoded columns)", oof, pred)

**+0.0015, the biggest gain of any feature idea on the ladder.** Encodings of how numbers are written beat any tuning you could do
on the raw columns.

<a id="s8"></a>
## 8 · Seed bagging: average away the encoding noise

The inner split of the encoding is random. A value seen 3 times gets a noticeably different training encoding
under a different split, and the model learns a little of that noise. So we rerun the same model with
other seeds (new inner splits and new tree subsampling, same outer folds) and average. The step-4 model is
seed 0.

In [ ]:
bag_oof, bag_pred, seed_scores = [OOF["lgb4_digits"]], [PRED["lgb4_digits"]], [roc_auc_score(y, OOF["lgb4_digits"])]
for seed in CFG.BAG_SEEDS[1:]:
    o, p = cv_lgb(te_view(ALL_KEYS), seeds=[seed])
    bag_oof.append(o)
    bag_pred.append(p)
    seed_scores.append(roc_auc_score(y, o))
log_step(5, "lgb4_digits_bag", f"+ seed bagging ({len(CFG.BAG_SEEDS)} encoding seeds)",
         np.mean(bag_oof, axis=0), np.mean(bag_pred, axis=0))
display(pd.DataFrame({"seed": CFG.BAG_SEEDS + ["bag (mean)"], "CV AUC": seed_scores + [LADDER[-1]["CV AUC"]]})
        .style.format({"CV AUC": "{:.5f}"}).hide(axis="index"))

The bag beats every single seed. The gain comes from removing noise, not from a lucky seed. The spread
between seeds is also a useful ruler: **a change smaller than that spread is not a result.**

<a id="s9"></a>
## 9 · Boosting from a GLM: small trees fix a linear model's mistakes 🆕

Section 5.3 showed that the labels behave like a logistic formula. A **GLM** (logistic regression) on the
encodings gets that additive part right, but it cannot bend. Trees can bend, but they spend most of their
splits re-learning the additive part. So we let each do its own job ([megayak's idea](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack)):

1. Fit a GLM on the **logits** of the encodings, plus one-hot categoricals, the flags and log-scaled numbers.
   The logit puts each encoding on the log-odds scale, where a linear model can add them up.
2. Give a 4-leaf LightGBM the GLM's logit as its **starting score** (`init_score`), so its trees fit only what the
   GLM gets wrong.

```
final logit  =  GLM logit (the additive part)  +  4-leaf trees (the bends and interactions)
```

Two details make the GLM stronger:

- **13 pair keys** (merchant category × risk score, foreign × IP mismatch, ...), target-encoded like every
  other key. A GLM is additive: on its own it cannot see that two columns *together* behave differently from the
  sum of their parts. Only the GLM gets these; the trees can build pairs themselves.
- **Solve it to convergence.** `LogisticRegression`'s default solver (lbfgs) stopped after 34 iterations here,
  before converging. Newton's method (`solver="newton-cholesky"`) converges in about 10 iterations, in the same
  time, and scores higher. Check `n_iter_` before you trust a linear model.

Inside each outer fold the GLM is fitted on that fold's training rows only, so validation and test rows start
from honest scores.

In [ ]:
# Pairs of columns the additive GLM cannot combine on its own ("q:" = deciles of a float column)
PAIRS = [("merchant_category", "merchant_risk_score"), ("is_foreign_transaction", "ip_country_mismatch"),
         ("merchant_category", "q:amount_usd"), ("txn_count_last_24h", "q:velocity_score"),
         ("merchant_category", "customer_age"), ("q:amount_usd", "q:merchant_risk_score"),
         ("q:hours_since_last_txn", "q:velocity_score"), ("customer_age", "q:merchant_risk_score"),
         ("is_new_merchant", "day_of_week"), ("is_ai_generated_scam_attempt", "q:velocity_score"),
         ("customer_age", "card_age_months"), ("time_of_day_hour", "day_of_week"),
         ("txn_count_last_24h", "customer_age")]
both = pd.concat([train[FEATURES], test[FEATURES]], ignore_index=True)


def pair_part(col):
    return pd.qcut(both[col[2:]], 10, labels=False, duplicates="drop") if col.startswith("q:") else both[col]


pairs = pd.DataFrame({f"{a}*{b}".replace("q:", ""): pair_part(a).astype(str) + "|" + pair_part(b).astype(str)
                      for a, b in PAIRS})
PAIR_KEYS = list(pairs.columns)
pair_codes, pair_levels = factorize(pairs.iloc[:len(train)], pairs.iloc[len(train):], PAIR_KEYS)
CODES.update(pair_codes)
N_LEVELS.update(pair_levels)
GLM_KEYS = ALL_KEYS + PAIR_KEYS
del both, pairs
print(f"{len(PAIR_KEYS)} pair keys for the GLM · {len(GLM_KEYS)} encoded columns in the GLM")


def cv_glm_boost(seeds=(0,), m=50, C=0.3):
    """Per fold: a GLM on the encoding logits, then 4-leaf LightGBM trees that start from the GLM's logit.
    Returns the out-of-fold and test predictions of both, averaged over seeds."""
    to_logit = lambda E: logit(np.clip(E.to_numpy(np.float64), 1e-5, 1 - 1e-5))
    log_num = lambda df: np.log1p(df[NUM].to_numpy(float))
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    out = {key: [np.zeros(len(train)), np.zeros(len(test))] for key in ("glm", "boost")}
    for seed in seeds:
        for k, (tr, va) in enumerate(FOLDS):
            # 1. the GLM, on this fold's training rows (smoothing 50 suits a linear model)
            E_tr, E_va, E_te = target_encode_fold(GLM_KEYS, tr, va, seed, m)
            sc_e = StandardScaler().fit(to_logit(E_tr))
            sc_n = StandardScaler().fit(log_num(train.iloc[tr]))
            oh = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(train[CAT].iloc[tr])
            design = lambda E, df: np.hstack([sc_e.transform(to_logit(E)), oh.transform(df[CAT]),
                                              sc_n.transform(log_num(df)), df[BOOL].to_numpy(float)])
            glm = LogisticRegression(C=C, solver="newton-cholesky", tol=1e-8, max_iter=200)
            glm.fit(design(E_tr, train.iloc[tr]), y[tr])
            z_tr, z_va, z_te = (glm.decision_function(design(E, df)) for E, df in
                                [(E_tr, train.iloc[tr]), (E_va, train.iloc[va]), (E_te, test)])
            # 2. 4-leaf trees on the step-4 features, starting from the GLM's logit
            X_tr, X_va, X_te = te_view(ALL_KEYS)(tr, va, seed)
            d_tr = lgb.Dataset(X_tr, y[tr], init_score=z_tr)
            d_va = lgb.Dataset(X_va, y[va], init_score=z_va, reference=d_tr)
            booster = lgb.train({**LGB_PARAMS, "seed": seed}, d_tr, num_boost_round=30_000, valid_sets=[d_va],
                                callbacks=[lgb.early_stopping(300, verbose=False)])
            b_va = sigmoid(z_va + booster.predict(X_va, num_iteration=booster.best_iteration, raw_score=True))
            b_te = sigmoid(z_te + booster.predict(X_te, num_iteration=booster.best_iteration, raw_score=True))
            for key, p_va, p_te in [("glm", sigmoid(z_va), sigmoid(z_te)), ("boost", b_va, b_te)]:
                out[key][0][va] += p_va / len(seeds)
                out[key][1] += p_te / (len(FOLDS) * len(seeds))
            print(f"  seed {seed} fold {k}: GLM {roc_auc_score(y[va], z_va):.5f} (converged in {glm.n_iter_[0]} "
                  f"iterations) -> GLM + trees {roc_auc_score(y[va], b_va):.5f} · {booster.best_iteration} trees")
    return out

In [ ]:
res = cv_glm_boost(seeds=CFG.BAG_SEEDS)
OOF["glm_bag"], PRED["glm_bag"] = res["glm"]
print(f"GLM alone, {len(CFG.BAG_SEEDS)} seeds: CV AUC {roc_auc_score(y, OOF['glm_bag']):.5f}")
log_step(6, "glmboost_bag", "+ boosting from a GLM (trees start at its logit)", *res["boost"])

**The GLM alone matches the step-5 bag** (0.95055 vs 0.95051), remarkable for a linear model: the encodings did
the non-linear work, and the GLM adds them up the way the original formula did. **The trees on top add another
+0.0005**, the biggest gain since the digit keys. This is the best single model in the notebook.

<a id="s10"></a>
## 10 · Blending, judged honestly

AUC only sees order, so we blend **ranks**, not probabilities: each member's predictions become ranks
scaled to [0, 1]. Members on different probability scales then count equally. First, how different are the
members?

In [ ]:
MEMBERS = {"glmboost_bag": "GLM -> 4-leaf trees · bag (step 6)",
           "lgb4_digits_bag": "LightGBM 4 leaves · digits · bag (step 5)",
           "lgb4_raw": "LightGBM 4 leaves · raw columns (step 2)"}


def to_rank(p):
    return rankdata(p) / len(p)


R = pd.DataFrame({m: to_rank(OOF[m]) for m in MEMBERS})
R_test = pd.DataFrame({m: to_rank(PRED[m]) for m in MEMBERS})
summary = pd.DataFrame({"model": MEMBERS.values(), "CV AUC": [roc_auc_score(y, OOF[m]) for m in MEMBERS]}, index=list(MEMBERS))
display(summary.style.format({"CV AUC": "{:.5f}"}))
print("Rank correlation between members:")
display(R.corr().style.format("{:.4f}").background_gradient(cmap="Blues", vmin=0.95, vmax=1))

The boosted GLM and the step-5 bag agree at 0.990. The raw-column model is the most different (0.964 to 0.967),
and that difference is what a blend can use.

Two ways to combine them:

- **Rank mean**: equal weights, nothing to fit, nothing to overfit.
- **Stack**: a logistic regression on the logits of the ranks learns one weight per member. To score it
  honestly, the stacker is cross-validated on a **second, different** 5-fold split, so it never scores rows it
  was fitted on.

In [ ]:
blend_oof = {"rank mean": R.mean(axis=1).to_numpy()}
Z = logit(R.clip(1e-6, 1 - 1e-6))
Z_test = logit(R_test.clip(1e-6, 1 - 1e-6))
stack_oof = np.zeros(len(y))
for tr, va in StratifiedKFold(5, shuffle=True, random_state=CFG.SEED + 1).split(Z, y):
    stack_oof[va] = LogisticRegression(max_iter=1000).fit(Z.iloc[tr], y[tr]).predict_proba(Z.iloc[va])[:, 1]
blend_oof["stack"] = stack_oof
stacker = LogisticRegression(max_iter=1000).fit(Z, y)

blend_test = {"rank mean": R_test.mean(axis=1).to_numpy(), "stack": stacker.predict_proba(Z_test)[:, 1]}
scores = {k: roc_auc_score(y, v) for k, v in blend_oof.items()}
display(pd.DataFrame({"CV AUC": scores}).style.format("{:.5f}"))
print("stack weights:", {m: round(float(w), 3) for m, w in zip(MEMBERS, stacker.coef_[0])})

In [ ]:
best_blend = max(scores, key=scores.get)
log_step(7, "blend", f"{best_blend} of {len(MEMBERS)} members", blend_oof[best_blend], blend_test[best_blend])
save_submission(blend_test[best_blend], "submission.csv")

This time the stack clearly beats the plain rank mean (0.95110 vs 0.95080). With members this unequal, equal
weights give the weaker ones too much say. The stack puts most weight on the boosted GLM, a fair share on the
step-5 bag, and a little on the raw-column model, whose mistakes differ most from the others'.

`submission.csv` is the final blend. The other two files are the Part A and Part B checkpoints.

> 📦 **Build on these predictions.** Every model's out-of-fold and test predictions are saved as
> `oof_predictions.csv` and `test_predictions.csv`, made on `StratifiedKFold(5, shuffle=True, random_state=42)`.
> If you use the same folds, you can stack them with your own models, or check where your model's mistakes differ.

In [ ]:
oof_frame = pd.DataFrame({ID: train[ID], TARGET: y, **OOF})
oof_frame.to_csv("oof_predictions.csv", index=False, float_format="%.7g")
pd.DataFrame({ID: test[ID], **PRED}).to_csv("test_predictions.csv", index=False, float_format="%.7g")
print(f"saved oof_predictions.csv and test_predictions.csv: {list(OOF)}")
del oof_frame

<a id="s11"></a>
## 11 · How noisy is the public leaderboard?

The public leaderboard scores about **20% of the test rows** (about 60,000 rows, roughly 1,000 frauds). How
much can that sample move a score? We can measure it on our own OOF predictions: draw many random
public-sized subsets and score each one.

In [ ]:
rng = np.random.default_rng(CFG.SEED)
n_public = round(0.2 * len(test))
step_models = [row["Model"] for row in LADDER]
samples = [rng.choice(len(y), n_public, replace=False) for _ in range(300)]
public = pd.DataFrame([[roc_auc_score(y[idx], OOF[m][idx]) for m in step_models] for idx in samples],
                      columns=step_models)
final = OOF["blend"]
pub, gap = public["blend"].to_numpy(), (public["blend"] - public["lgb4_digits"]).to_numpy()

fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
axes[0].hist(pub, bins=30, color=BLUE, edgecolor=SURFACE, linewidth=0.8)
axes[0].axvline(roc_auc_score(y, final), color=INK, lw=1.5)
axes[0].set_title("The final blend, scored on 300 random public-sized samples")
axes[0].set_xlabel("AUC on a 20% sample")
axes[1].hist(gap * 1e4, bins=30, color=BLUE, edgecolor=SURFACE, linewidth=0.8)
axes[1].axvline(0, color=INK, lw=1.5)
axes[1].set_title("Final blend minus step-4 single model, same samples")
axes[1].set_xlabel("AUC difference x 1e-4")
for ax in axes:
    ax.grid(axis="x", visible=False)
plt.tight_layout()
plt.show()
print(f"one submission on a public-sized sample: AUC moves ±{pub.std():.4f} (1 sd); "
      f"90% range {np.quantile(pub, 0.05):.4f} – {np.quantile(pub, 0.95):.4f}")

# every step of the ladder against the one before it, on the same public-sized samples
adjacent = []
for prev, cur in zip(LADDER[:-1], LADDER[1:]):
    d = public[cur["Model"]] - public[prev["Model"]]
    adjacent.append({"step": f"{prev['Step']} → {cur['Step']}", "what changes": cur["What changes"],
                     "CV gain": cur["CV AUC"] - prev["CV AUC"], "public-sized: mean gain": d.mean(),
                     "sd": d.std(), "later step ahead": (d > 0).mean()})
display(pd.DataFrame(adjacent).style.format({"CV gain": "{:+.5f}", "public-sized: mean gain": "{:+.5f}",
                                             "sd": "{:.5f}", "later step ahead": "{:.0%}"}).hide(axis="index"))

Two lessons:

1. The public score of *one* submission swings by about **±0.003** (one standard deviation) depending on which rows
   happen to be public: anywhere from about 0.946 to 0.956 for the same model. That is bigger than every gain on
   our ladder after step 1, so **don't compare your CV with someone else's public score**.
2. Two submissions are scored on the *same* public rows, so their **difference** is much less noisy than either
   score. How much less depends on how alike the two models rank the rows (a point raised in the comments), so
   the table measures it for every step of the ladder:
   - the big steps (31 → 4 leaves, the encodings, the digit keys) win on 95–100% of the samples;
   - seed bagging (+0.0002 CV) wins on only about 4 samples in 5, although the gain is real;
   - the final stack's +0.00004 over step 6 is a coin flip: real on 700k rows, invisible on 60k.

> 💡 **Pick your final submissions by CV** (700k rows), not by the public board (60k rows). Use the board as a
> sanity check for big differences only.

One limit, also from the comments: these samples measure only *which rows* land in the public 20%. They take the
saved OOF predictions as given, so they say nothing about other sources of public-to-private shift.

This notebook's own submissions: version 2's blend scored 0.95083 on the public board (CV 0.95084), version 3's
scores 0.95110 (CV 0.95110).

---
<a id="s12"></a>
## 12 · From AUC to a fraud team's review queue

AUC is abstract. A fraud team thinks in **review budgets**: *if we can check the top X% of transactions, how
much fraud do we stop?* Out-of-fold, on 700k rows:

In [ ]:
order = np.argsort(-final)
rows = []
for share in [0.005, 0.01, 0.02, 0.05, 0.10]:
    k = round(share * len(y))
    hits = y[order[:k]].sum()
    rows.append({"review the top": f"{share:.1%}", "transactions": k, "frauds caught": hits,
                 "recall": hits / y.sum(), "precision": hits / k, "lift vs random": hits / k / y.mean()})
display(pd.DataFrame(rows).style.format({"transactions": "{:,}", "frauds caught": "{:,}", "recall": "{:.1%}",
                                         "precision": "{:.1%}", "lift vs random": "{:.1f}x"}).hide(axis="index"))

Reviewing the riskiest **1%** of transactions catches **40% of all fraud**, and two out of three reviewed
transactions really are fraud: 40 times better than random checks. Reviewing 5% catches almost three quarters.
That is what an AUC of 0.95 buys in practice.

---
<a id="partD"></a>
# 🏆 Part D · Optional: pooling public predictions, the way the top of the board is built

Everything above is trained in this notebook. The highest public scores are built differently. Many authors
publish their models' **out-of-fold and test predictions**, all made on the same `StratifiedKFold(5, shuffle=True,
random_state=42)` split as this notebook. A **pooled stack** collects them, adds your own models, and lets a small
logistic regression learn how much to trust each one.

Two rules keep it honest:

1. **Only pool predictions made on your folds.** Otherwise a "out-of-fold" prediction may have seen the row it
   predicts, and the stack learns a leak. Three of the sources below ship their fold assignments, and the code
   checks that they match ours.
2. **Score the stack on a second split**, exactly as in section 10, so it never scores rows it was fitted on.

The inputs are public notebooks and datasets, published so that others can pool them. Every source is credited in
the table below and in section 14. To run this part, attach them with *Add Input*; the cells skip themselves if they
are missing. Their authors may publish new versions, so your numbers can drift from the ones shown here.

| source | what it brings |
|---|---|
| goodpjw2008 · [S6E11 fraud series artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts) | GLM boosts, residual MLPs, EBMs, RealMLPs, digit-tree bags |
| masayakawamata · [GPT-2 target-first classifier](https://www.kaggle.com/code/masayakawamata/s6e11-gpt-2-target-first-classifier-cv-0-9500) | a GPT-2 model that reads each row as text |
| megayak · [Per-Value + Digit TE, Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack) | the original GLM boosts and digit trees |
| lazyyeou · [Digit-Trie TE, Joint GLM & Nested Stack](https://www.kaggle.com/code/lazyyeou/s6e11-digit-trie-te-joint-glm-nested-stack) | seed-bagged digit trees and a 96-key GLM |
| Malyshev Danil · [S6E11 pooled stack v9](https://www.kaggle.com/datasets/malyshevdanil/s6e11-pooled-stack-v9) | their own EBMs and enriched trees, plus Nilanjan_Saha123's 96-key models |
| Will · [S6E11 tree ensemble predictions](https://www.kaggle.com/datasets/wguesdon/s6e11-tree-ensemble-predictions) | raw and encoded GBDTs, a digit-pair stump |

In [ ]:
def find_dir(marker):
    """Folder of the first input file named `marker`, or None."""
    hit = find_file(marker)
    return hit.parent if hit is not None else None


def load(path):
    return np.load(path, allow_pickle=False).astype(np.float64).ravel()   # data only: never unpickle inputs


POOL = {}   # name -> (oof, test, source)
banks = {"goodpjw2008": find_dir("gb_bag5_oof.npy"), "masayakawamata": find_dir("fold_ids.npy"),
         "megayak": find_dir("oof_glm_boost.npy"), "lazyyeou": find_dir("lgb_l4_bag_oof.npy"),
         "malyshevdanil": find_dir("pool_bank.npz"), "wguesdon": find_dir("digit_pair_stump.npz")}
print({k: ("found" if v is not None else "missing") for k, v in banks.items()})
RUN_PART_D = any(v is not None for v in banks.values())

if RUN_PART_D:
    our_folds = np.zeros(len(y), int)
    for k, (_, va) in enumerate(FOLDS):
        our_folds[va] = k
    same_split = lambda f: pd.crosstab(our_folds, np.asarray(f).astype(int)).gt(0).sum(axis=1).eq(1).all()
    if (d := banks["goodpjw2008"]) is not None:
        for f in sorted(d.glob("*_oof.npy")):
            stem = f.name[:-8]
            t = d / f"{stem}_full.npy" if (d / f"{stem}_full.npy").exists() else d / f"{stem}_test.npy"
            if t.exists():
                POOL[f"good:{stem}"] = (load(f), load(t), "goodpjw2008")
    if (d := banks["masayakawamata"]) is not None:
        ok = same_split(np.load(d / "fold_ids.npy", allow_pickle=False))
        print("masayakawamata folds match ours:", ok)
        if ok:
            POOL["masaya:gpt2"] = (load(d / "oof.npy"), load(d / "test.npy"), "masayakawamata")
    if (d := banks["megayak"]) is not None:
        for f in sorted(d.glob("oof_*.npy")):
            POOL[f"megayak:{f.name[4:-4]}"] = (load(f), load(d / f"refit_{f.name[4:]}"), "megayak")
    if (d := banks["lazyyeou"]) is not None:
        for f in sorted(d.glob("*_oof.npy")):
            if (d / f"{f.name[:-8]}_full.npy").exists():
                POOL[f"lazy:{f.name[:-8]}"] = (load(f), load(d / f"{f.name[:-8]}_full.npy"), "lazyyeou")
    if (d := banks["malyshevdanil"]) is not None:
        names = __import__("json").load(open(d / "pool_manifest.json"))["names"]
        with np.load(d / "pool_bank.npz", allow_pickle=False) as z:
            assert (z["y"] == y).all()
            for j, n in enumerate(names):
                POOL[f"malyshev:{n}"] = (z["oof"][:, j].astype(np.float64), z["test"][:, j].astype(np.float64), "malyshevdanil")
    if (d := banks["wguesdon"]) is not None:
        f = pd.read_csv(d / "folds.csv")
        ok = same_split(f.set_index("id").loc[train[ID], "fold"])
        print("wguesdon folds match ours:", ok)
        for npz in sorted(d.glob("*.npz")) if ok else []:
            with np.load(npz, allow_pickle=False) as z:
                if "oof" in z.files:
                    assert (z["train_id"] == train[ID].to_numpy()).all() and (z["test_id"] == test[ID].to_numpy()).all()
                    POOL[f"wguesdon:{npz.stem}"] = (z["oof"].astype(np.float64), z["fold_test"].mean(axis=0), "wguesdon")
    for name in ["glmboost_bag", "glm_bag", "lgb4_digits_bag", "lgb4_te", "lgb4_raw", "lgb31_raw", "logreg_raw"]:
        POOL[f"ours:{name}"] = (OOF[name], PRED[name], "this notebook")
    assert all(len(o) == len(y) and len(t) == len(test) for o, t, _ in POOL.values())
    print(f"{len(POOL)} candidate models")

Many models appear in more than one bank (Malyshev's bank includes copies of lazyyeou's and goodpjw2008's), so we
keep one copy of every model (rank correlation above 0.99995 counts as a copy) and drop the weak ones (AUC below
0.945). Then the same stack as section 10: a logistic regression on standardized logits of the ranks, scored on a
second 5-fold split. With ~80 correlated inputs the penalty matters, so we try three strengths.

In [ ]:
if RUN_PART_D:
    ranks = {n: to_rank(v[0]) for n, v in POOL.items()}
    pool_auc = pd.Series({n: roc_auc_score(y, v[0]) for n, v in POOL.items()}).sort_values(ascending=False)
    kept = []
    for n in pool_auc.index:
        if pool_auc[n] >= 0.945 and all(np.corrcoef(ranks[n], ranks[k])[0, 1] <= 0.99995 for k in kept):
            kept.append(n)
    src = pd.Series({n: POOL[n][2] for n in kept})
    print(f"{len(kept)} distinct models kept:", src.value_counts().to_dict())

    ZP = logit(np.clip(np.column_stack([ranks[n] for n in kept]), 1e-6, 1 - 1e-6))
    ZP_test = logit(np.clip(np.column_stack([to_rank(POOL[n][1]) for n in kept]), 1e-6, 1 - 1e-6))
    mu, sd = ZP.mean(axis=0), ZP.std(axis=0)
    ZP, ZP_test = (ZP - mu) / sd, (ZP_test - mu) / sd
    second_split = list(StratifiedKFold(5, shuffle=True, random_state=CFG.SEED + 1).split(ZP, y))

    def pooled_cv(cols, C):
        oof = np.zeros(len(y))
        for a, b in second_split:
            m = LogisticRegression(C=C, solver="newton-cholesky", tol=1e-8, max_iter=200).fit(ZP[a][:, cols], y[a])
            oof[b] = m.decision_function(ZP[b][:, cols])
        return oof

    everything = list(range(len(kept)))
    pooled = {C: pooled_cv(everything, C) for C in [0.01, 0.1, 1.0]}
    pooled_scores = {C: roc_auc_score(y, o) for C, o in pooled.items()}
    display(pd.DataFrame({"nested CV AUC": pooled_scores}).rename_axis("C").style.format("{:.5f}"))
    best_C = max(pooled_scores, key=pooled_scores.get)

What does each source add? Remove one source at a time and re-score the stack:

In [ ]:
if RUN_PART_D:
    full_score = pooled_scores[best_C]
    loso = []
    for source in src.unique():
        cols = [i for i, n in enumerate(kept) if src[n] != source]
        loso.append({"without": source, "models removed": int((src == source).sum()),
                     "nested CV AUC": roc_auc_score(y, pooled_cv(cols, best_C)),})
    loso = pd.DataFrame(loso)
    loso["what it adds"] = full_score - loso["nested CV AUC"]
    display(loso.sort_values("what it adds", ascending=False).style.format(
        {"nested CV AUC": "{:.5f}", "what it adds": "{:+.5f}"}).hide(axis="index"))

    stacker_p = LogisticRegression(C=best_C, solver="newton-cholesky", tol=1e-8, max_iter=200).fit(ZP, y)
    sigmoid = lambda z: 1 / (1 + np.exp(-z))
    log_step(8, "pooled", f"pooled stack of {len(kept)} models (public + ours)", sigmoid(pooled[best_C]),
             sigmoid(stacker_p.decision_function(ZP_test)))
    save_submission(PRED["pooled"], "submission_pooled.csv")
    weights = pd.Series(stacker_p.coef_[0], index=kept).sort_values(ascending=False)
    print("largest weights:", weights.head(5).round(3).to_dict())

Three things stand out:

- **The GPT-2 classifier adds the most** (about +0.0004), although its own score is not the highest. It reads each
  row as text, so its mistakes differ from every tree and GLM in the pool. Diversity beats strength in a stack.
- **This notebook's boosted GLM earns one of the largest weights**, right behind GPT-2. Yet removing all of this
  notebook's models costs only 0.00002: goodpjw2008's and megayak's GLM boosts cover the same ground.
- **Tree models are saturated**: dozens of shallow-tree variants from five authors add little over each other.

`submission_pooled.csv` is the pooled stack: nested CV 0.95214, public LB 0.95200 when this section was written.
`submission.csv` stays the self-trained blend of Part C.

[Back to contents](#contents)

<a id="s13"></a>
## 13 · What didn't work, and what to try next

**Measured while building this notebook.** Changes to the step-4 model, on fold 0 (step-4 fold-0 AUC 0.94939;
not rerun here, to save time). Read anything within about ±0.0002 as noise: single seeds of the step-4 model
differ by up to 0.00026 over all five folds (section 8), and one fold is noisier still.

| idea | fold-0 AUC change | note |
|---|---|---|
| 7 leaves instead of 4 | −0.00057 | still too deep |
| 3 leaves instead of 4 | +0.00005 | noise; 3 or 4 both fine |
| digit keys without `lead1` / `lead2` | −0.00032 | the leading digits matter |
| learning rate 0.03 instead of 0.05 | −0.00002 | 45% more trees, no better |
| smoothing 50 instead of 20 | +0.00005 | noise; 20 kept |

**Version 3 experiments** (fold 0 unless noted):

| idea | result | note |
|---|---|---|
| XGBoost depth 2 as a stack member (versions 1–2) | 5-fold 0.95051, same as the LightGBM bag (rank correlation 0.997) | added nothing to the v3 stack; dropped, saving about 23 minutes per run |
| GLM solved to convergence (Newton) instead of lbfgs's default stop | GLM +0.00008, boosted +0.00007 | lbfgs stopped after 34 iterations |
| 13 pair keys in the GLM | GLM +0.00035, boosted +0.00011 | kept |
| value counts of every key as features for the boosted trees | −0.00005 to −0.0002 | dropped |
| learning rate 0.02 instead of 0.05 for the boosted trees | +0.00003 | twice the trees; 0.05 kept |

And from the ladder itself: 31 leaves instead of 4 cost 0.00108 (5-fold, steps 1 and 2), and an encoding fitted
before the split "scored" 0.98 on fold 0 (section 6.3).

**Reported by others** (their notebooks are in the credits):

| idea | reported result |
|---|---|
| appending the 20k original rows to training | −0.0003 to ±0 (the original rows are distributed differently) |
| target encodings of column pairs | −0.0006 to −0.0007 (shallow trees already model pairs) |
| numeric columns as native LightGBM categoricals | −0.0054 (per-value splits overfit; smoothed encodings work) |
| GPT-2 token keys on top of the digit keys | −0.0002 to +0.0001: no clear gain |
| 10 inner encoding folds instead of 5 | −0.0003 to −0.0006 (the noise of 5 folds regularizes) |

**Ideas to try next**

- **Refit on all training rows** for the test predictions (encodings fitted on 100% of the rows instead of 80%);
  others report small public gains with mixed CV evidence.
- **More diverse members**: RealMLP (`pytabkit`), Explainable Boosting Machines, or residual MLPs that start from
  the GLM's logit like the trees of section 9. The strongest new family on the board is a target-first GPT-2
  classifier, at a cost of about 120 GPU hours.
- **Pooling public predictions**: many notebooks publish out-of-fold predictions on this same split (this one
  included). Stacking them with nested CV is how the top public scores are built.
- **More seeds**: gains flatten after roughly 5–8.
- **Final selection**: choose 2 submissions. A good pair is your CV-best blend plus a robust, simpler one.

<a id="s14"></a>
## 14 · Credits

This notebook stands on public work from the S6E11 community. Please upvote the originals too:

- **Udit Jain**, [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026): the original dataset.
- **megayak (Artificial Idiocy Lab)**, [S6E11: Per-Value + Digit TE, Shallow-Tree Stack](https://www.kaggle.com/code/megayak/s6e11-per-value-digit-te-shallow-tree-stack): boosting from the GLM and the pair keys of section 9.
- **goodpjw2008**, [S6E11 Part 8: Two Stacks, One Blend](https://www.kaggle.com/code/goodpjw2008/s6e11-part-8-two-stacks-one-blend-lb-0-95235): the digit keys, tiny trees and the plain GLM, and the careful measurements of the GLM boost (its Part 6), documented step by step.
- **SIDHAARTH SHREE**, [S6E11: Highest Public LB, Self-Trained](https://www.kaggle.com/code/sidhaarthshree/s6e11-highest-public-lb-0-95154-self-trained): per-value encoding and the key list.
- **Tudor Beloiu**, [Depth-2 Trees + Digit TE, Seed-Bagged](https://www.kaggle.com/code/tudorbeloiu/depth-2-trees-digit-te-seed-bagged-lb-0-95118): the "truncate, don't round" detail and seed averaging of the encoder.
- **Areeb Ali**, [S6E11: Pooled Stack + Fusion](https://www.kaggle.com/code/areebbb/s6e11-pooled-stack-fusion-lb-0-95178): seed bagging of target encodings, measured fold by fold.
- **Дворкин Евгений Владимирович**, [S6E11: Deep EDA, Original Data & Artifacts](https://www.kaggle.com/code/evgendvorkin/s6e11-deep-eda-original-data-artifacts): how synthetic values relate to the original data.
- **aphoristicartist**, [S6E11 Fraud: What Predicts It and What Doesn't](https://www.kaggle.com/code/aphoristicartist/s6e11-fraud-what-predicts-it-and-what-doesn-t): how the risk signals compound.

- **Part D's sources**: goodpjw2008, masayakawamata (GPT-2 classifier, CC BY 4.0), megayak, lazyyeou, Malyshev Danil
  (with Nilanjan_Saha123's models) and Will, for publishing their predictions on the shared folds.
- **Georgy Mamarin**, in the comments: the step-by-step noise table of section 11, and its limits.

If this notebook helped you, an upvote helps others find it. Questions, corrections and ideas are very
welcome in the comments.

<a id="s15"></a>
## 15 · The ladder, recomputed live

In [ ]:
ladder = pd.DataFrame(LADDER)
ladder["gain"] = ladder["CV AUC"].diff()
display(ladder.style.format({"CV AUC": "{:.5f}", "gain": "{:+.5f}"}, na_rep="").hide(axis="index"))

fig, ax = plt.subplots(figsize=(10, 4.4))
yy = np.arange(len(ladder))[::-1]
ax.plot(ladder["CV AUC"], yy, color=GRID, lw=2, zorder=1)
ax.scatter(ladder["CV AUC"], yy, s=55, color=BLUE, edgecolor=SURFACE, linewidth=1.5, zorder=3)
for v, row in zip(ladder["CV AUC"], yy):
    ax.annotate(f"{v:.5f}", (v, row), xytext=(8, 0), textcoords="offset points", va="center", fontsize=8.5, color=INK_2)
ax.set_yticks(yy, [f"{s} · {w}" for s, w in zip(ladder["Step"], ladder["What changes"])])
ax.set_xlim(ladder["CV AUC"].min() - 0.0008, ladder["CV AUC"].max() + 0.0012)
ax.set_xlabel("5-fold CV AUC")
ax.grid(axis="y", visible=False)
ax.set_title("Every step of the ladder, measured on the same 5 folds")
plt.tight_layout()
plt.show()
print(f"total runtime: {(time.time() - T_START) / 60:.1f} min")